# Rollout corregido de la política del G1

Receta mínima para ejecutar bien la política `2026-01-26_10-42-02.onnx` sobre el Unitree G1.

El problema no era la red en sí, sino varios detalles alrededor: modelo incompleto, pose inicial incorrecta, observación distinta a la de entrenamiento y acciones escritas en el orden equivocado.

## 1. Rutas y política local

Usamos directamente el `.onnx` local.

In [9]:
from pathlib import Path
import os
import re
import sys

os.environ.setdefault('MUJOCO_GL', 'egl')

import mediapy as media
import mujoco
import numpy as np
import onnxruntime as ort


def find_workspace_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/home/omm/Projects/diego_fix')]
    for path in candidates:
        if (path / 'mjlab/src/mjlab').exists() and (path / '2025-tfg-diego-lopez/pruebas/G1').exists():
            return path
    raise RuntimeError('No se encontró la raíz del proyecto diego_fix')


WORKSPACE = find_workspace_root()
NOTEBOOK_DIR = WORKSPACE / '2025-tfg-diego-lopez/pruebas/G1'
MJLAB_SRC = WORKSPACE / 'mjlab/src'
POLICY_PATH = NOTEBOOK_DIR / '2026-01-26_10-42-02.onnx'

if str(MJLAB_SRC) not in sys.path:
    sys.path.insert(0, str(MJLAB_SRC))

assert POLICY_PATH.exists(), POLICY_PATH

## 2. Constantes del G1 que sí corresponden

El `g1_constants.py` local no cuadraba del todo con `mjlab`. Importamos las constantes activas: XML, colisiones, pose inicial, actuadores y escalas de acción.

In [10]:
from mjlab.asset_zoo.robots.unitree_g1.g1_constants import (
    FULL_COLLISION,
    G1_ACTION_SCALE,
    G1_ACTUATOR_4010,
    G1_ACTUATOR_5020,
    G1_ACTUATOR_7520_14,
    G1_ACTUATOR_7520_22,
    G1_ACTUATOR_ANKLE,
    G1_ACTUATOR_WAIST,
    G1_XML,
    KNEES_BENT_KEYFRAME,
)

TIMESTEP = 0.005
DECIMATION = 4
OBS_DIM = 99
ACTION_DIM = 29
FREE_QPOS = 7
FREE_QVEL = 6

ACTUATOR_CFGS = (
    G1_ACTUATOR_5020,
    G1_ACTUATOR_7520_14,
    G1_ACTUATOR_7520_22,
    G1_ACTUATOR_4010,
    G1_ACTUATOR_WAIST,
    G1_ACTUATOR_ANKLE,
)
ACTUATOR_GROUPS = [
    (cfg.target_names_expr, cfg.stiffness, cfg.damping, cfg.effort_limit, cfg.armature)
    for cfg in ACTUATOR_CFGS
]
INIT_ROOT_POS = np.array(KNEES_BENT_KEYFRAME.pos, dtype=np.float32)
INIT_JOINT_POS = KNEES_BENT_KEYFRAME.joint_pos

## 3. Construcción del modelo MuJoCo

El XML base no trae todo lo que necesita la política para este rollout: añadimos suelo, colisiones y actuadores de posición.

La keyframe `init_state` evita otro fallo importante: `mj_resetData` deja las articulaciones a cero, pero la política espera la pose inicial del G1.

In [11]:
def spec_joint_names(spec: mujoco.MjSpec) -> list[str]:
    return [
        joint.name
        for joint in spec.joints
        if joint.type != mujoco.mjtJoint.mjJNT_FREE
    ]


def resolve_joint_positions(joint_names: list[str]) -> np.ndarray:
    values = np.zeros(len(joint_names), dtype=np.float32)
    for pattern, value in INIT_JOINT_POS.items():
        for i, name in enumerate(joint_names):
            if re.fullmatch(pattern, name):
                values[i] = value
    return values


def matching_names(patterns: tuple[str, ...], names: list[str]) -> list[str]:
    return [
        name
        for name in names
        if any(re.fullmatch(pattern, name) for pattern in patterns)
    ]


def add_position_actuator(spec, joint_name, stiffness, damping, effort_limit, armature):
    actuator = spec.add_actuator(name=joint_name, target=joint_name)
    actuator.trntype = mujoco.mjtTrn.mjTRN_JOINT
    actuator.dyntype = mujoco.mjtDyn.mjDYN_NONE
    actuator.gaintype = mujoco.mjtGain.mjGAIN_FIXED
    actuator.biastype = mujoco.mjtBias.mjBIAS_AFFINE
    actuator.gainprm[0] = stiffness
    actuator.biasprm[1] = -stiffness
    actuator.biasprm[2] = -damping
    actuator.inheritrange = 0.0
    actuator.ctrllimited = False
    actuator.forcelimited = True
    actuator.forcerange[:] = np.array([-effort_limit, effort_limit])
    spec.joint(joint_name).armature = armature


def build_g1_model() -> tuple[mujoco.MjModel, mujoco.MjData]:
    spec = mujoco.MjSpec.from_file(str(G1_XML))
    joint_names = spec_joint_names(spec)
    default_joint_pos = resolve_joint_positions(joint_names)

    FULL_COLLISION.edit_spec(spec)
    spec.worldbody.add_geom(
        name='floor',
        type=mujoco.mjtGeom.mjGEOM_PLANE,
        size=[10, 10, 0.1],
        pos=[0, 0, 0],
    )

    name_to_default = dict(zip(joint_names, default_joint_pos))
    ctrl_default = []
    for patterns, stiffness, damping, effort_limit, armature in ACTUATOR_GROUPS:
        for joint_name in matching_names(patterns, joint_names):
            add_position_actuator(spec, joint_name, stiffness, damping, effort_limit, armature)
            ctrl_default.append(name_to_default[joint_name])

    key_qpos = np.concatenate([
        INIT_ROOT_POS,
        np.array([1, 0, 0, 0], dtype=np.float32),
        default_joint_pos,
    ])
    spec.add_key(name='init_state', qpos=key_qpos.tolist(), ctrl=ctrl_default)

    model = spec.compile()
    model.opt.timestep = TIMESTEP
    model.opt.iterations = 10
    model.opt.ls_iterations = 20
    if hasattr(model.opt, 'ccd_iterations'):
        model.opt.ccd_iterations = 500
    return model, mujoco.MjData(model)


model, data = build_g1_model()
print('Actuadores:', model.nu)
print('qpos:', model.nq, 'qvel:', model.nv)

Actuadores: 29
qpos: 36 qvel: 35


## 4. Carga de la política

Comprobación rápida de interfaz: la política debe recibir 99 valores y devolver 29 acciones.

In [12]:
session = ort.InferenceSession(str(POLICY_PATH), providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

print(input_name, session.get_inputs()[0].shape)
print(output_name, session.get_outputs()[0].shape)

assert session.get_inputs()[0].shape == [1, OBS_DIM]
assert session.get_outputs()[0].shape == [1, ACTION_DIM]

obs [1, 99]
actions [1, 29]


## 5. Órdenes de articulaciones y actuadores

Este era uno de los fallos clave. La política devuelve acciones en orden de articulaciones; `data.ctrl` espera orden de actuadores. No coinciden.

Por eso convertimos siempre los objetivos antes de escribir en `data.ctrl`.

In [13]:
def non_free_joint_names(model: mujoco.MjModel) -> list[str]:
    names = []
    for joint_id in range(model.njnt):
        if model.jnt_type[joint_id] != mujoco.mjtJoint.mjJNT_FREE:
            names.append(mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, joint_id))
    return names


def actuator_names(model: mujoco.MjModel) -> list[str]:
    return [
        mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
        for i in range(model.nu)
    ]


def action_scale_for_joint(joint_name: str) -> float:
    for pattern, scale in G1_ACTION_SCALE.items():
        if re.fullmatch(pattern, joint_name):
            return float(scale)
    raise KeyError(f'No hay escala de acción para {joint_name}')


joint_order = non_free_joint_names(model)
ctrl_order = actuator_names(model)
joint_index = {name: i for i, name in enumerate(joint_order)}
action_scale = np.array([action_scale_for_joint(name) for name in joint_order], dtype=np.float32)
default_joint_pos = model.key('init_state').qpos[FREE_QPOS:].astype(np.float32).copy()


def joint_targets_to_ctrl(joint_targets: np.ndarray) -> np.ndarray:
    return np.array([joint_targets[joint_index[name]] for name in ctrl_order], dtype=np.float32)


print('Primeras articulaciones:', joint_order[:6])
print('Primeros actuadores:', ctrl_order[:6])

Primeras articulaciones: ['left_hip_pitch_joint', 'left_hip_roll_joint', 'left_hip_yaw_joint', 'left_knee_joint', 'left_ankle_pitch_joint', 'left_ankle_roll_joint']
Primeros actuadores: ['left_shoulder_pitch_joint', 'left_shoulder_roll_joint', 'left_shoulder_yaw_joint', 'left_elbow_joint', 'left_wrist_roll_joint', 'right_shoulder_pitch_joint']


## 6. Observación de la política

La observación tiene que respetar el orden y el significado usados al entrenar. Los detalles importantes son: posiciones relativas, gravedad en la pelvis y `last_action` persistente.

1. velocidad lineal del IMU,
2. velocidad angular del IMU,
3. gravedad proyectada en la pelvis,
4. posiciones relativas de articulaciones,
5. velocidades de articulaciones,
6. acción anterior sin modificar,
7. comando de velocidad.

In [14]:
def sensor_slice(model: mujoco.MjModel, sensor_name: str) -> slice:
    sensor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SENSOR, sensor_name)
    start = model.sensor_adr[sensor_id]
    return slice(start, start + model.sensor_dim[sensor_id])


lin_vel_slice = sensor_slice(model, 'imu_lin_vel')
ang_vel_slice = sensor_slice(model, 'imu_ang_vel')
pelvis_body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, 'pelvis')


def projected_gravity(data: mujoco.MjData) -> np.ndarray:
    pelvis_xmat = data.xmat[pelvis_body_id].reshape(3, 3)
    return pelvis_xmat.T @ np.array([0.0, 0.0, -1.0], dtype=np.float32)


def make_observation(data, last_action, command) -> np.ndarray:
    obs = np.concatenate([
        data.sensordata[lin_vel_slice],
        data.sensordata[ang_vel_slice],
        projected_gravity(data),
        data.qpos[FREE_QPOS:] - default_joint_pos,
        data.qvel[FREE_QVEL:],
        last_action,
        command,
    ]).astype(np.float32)
    assert obs.shape == (OBS_DIM,), obs.shape
    return obs

## 7. Inferencia y aplicación de acciones

No recortamos la salida de la red. Cliparla a `[-1, 1]` hacía que el robot acabara cayendo: la política necesita usar su acción cruda, escalada alrededor de la pose inicial.

In [15]:
def reset_to_default():
    key_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_KEY, 'init_state')
    mujoco.mj_resetDataKeyframe(model, data, key_id)
    data.ctrl[:] = joint_targets_to_ctrl(default_joint_pos)
    mujoco.mj_forward(model, data)


def infer_action(last_action: np.ndarray, command: np.ndarray) -> np.ndarray:
    obs = make_observation(data, last_action, command)[None, :]
    return session.run([output_name], {input_name: obs})[0][0].astype(np.float32)


def apply_action(raw_action: np.ndarray):
    joint_targets = default_joint_pos + raw_action * action_scale
    data.ctrl[:] = joint_targets_to_ctrl(joint_targets)

## 8. Rollout sencillo

Bucle único, sin hilos. Cada inferencia se mantiene durante 4 pasos de MuJoCo, igual que en el entrenamiento. El resultado esperado es simplemente ver al robot caminar hacia delante.

In [16]:
duration_s = 6.0
framerate = 60
command = np.array([0.6, 0.0, 0.0], dtype=np.float32)

reset_to_default()
frames = []
last_action = np.zeros(ACTION_DIM, dtype=np.float32)
start_x = float(data.qpos[0])

with mujoco.Renderer(model) as renderer:
    while data.time < duration_s:
        raw_action = infer_action(last_action, command)
        apply_action(raw_action)
        last_action = raw_action

        for _ in range(DECIMATION):
            mujoco.mj_step(model, data)
            if len(frames) < data.time * framerate:
                renderer.update_scene(data)
                frames.append(renderer.render())

print(f'Avance aproximado: {data.qpos[0] - start_x:.2f} m')
media.show_video(frames, fps=framerate)

Avance aproximado: 3.02 m


This browser does not support the video tag.
